# Task 1: Summarization as Memory (S2)
**Owner:** Sai Prasad Bandari · **Team:** AIS 6

## 1. Research Context & Connection
- **Papers:** `#3` ACON (threshold-triggered context compression), `#4` ReSum (periodic summary, then continue from it), `#18` Recursive Summarization (Wang et al., Neurocomputing 2025), `#1` LongMemEval (benchmark), `#15` Lost in the Middle.
- **Research Question:** When a long chat history (~115k tokens) must be shrunk to a fixed token budget (main budget: 8k tokens), which summarization method keeps the information needed to answer correctly, and at what cost in tokens and time?
- **Hypothesis:** Summaries lose fine details (dates, numbers), so they should lose against truncation on detail questions but win on questions that need the whole history.


In [ ]:
# 2. Setup & Reproducibility
import os, sys, random
import numpy as np
import torch

sys.path.append("..")
from src.config import SEED, READER_MODEL_ID, MAIN_TOKEN_BUDGET, SummarizationPrompts
from src.summarization import (
    truncate_latest_tokens,
    summarize_one_shot,
    summarize_recursive,
    summarize_hybrid_with_recent,
)
from src.judge import evaluate_answer_correctness, diagnose_lost_vs_ignored
from src.metrics import bootstrap_accuracy_ci, paired_mcnemar_test

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 3. Load Stratified 100-Question LongMemEval-S Subset


In [ ]:
from datasets import load_dataset
from src.data_loader import stratified_sample_longmemeval, format_session_turns, extract_evidence_texts

# Load LongMemEval-S and sample the shared 100-question stratified subset
print("Ready to load LongMemEval-S with SEED =", SEED)


## 4. Load 4-Bit Quantized Qwen2.5-7B-Instruct (Reader & Summarizer)
*Note: YaRN RoPE scaling is enabled only when inputs exceed the native 32k context window.*


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
print(f"Configured 4-bit NF4 quantization for {READER_MODEL_ID}")


## 5. Run Experimental Variations (a) One-Shot, (b) Recursive, (c) Summary + Recent vs. Truncation Reference
All variations are evaluated at the same target token budget (`MAIN_TOKEN_BUDGET = 8192`) with greedy decoding.


In [ ]:
# Execute and cache summaries to results/summaries/ for each variant:
# - Reference: truncate_latest_tokens(...)
# - Variant (a): summarize_one_shot(...)
# - Variant (b): summarize_recursive(...)
# - Variant (c): summarize_hybrid_with_recent(...)


## 6. Deeper Analysis: "Lost vs. Ignored" Diagnostic & Statistical Testing
For every wrong answer (score = 0), check whether the labeled evidence survived in the summary:
- **Evidence absent:** Lost by the summary.
- **Evidence present:** Ignored by the reader model.
Also compute 95% bootstrap confidence intervals and paired McNemar tests across variants.
